# Clipzi AI — Whisper model → Dropbox

Notebook ini **self-contained**: tidak perlu clone repo Clipzi yang private.

1. Isi kredensial Dropbox di sel konfigurasi (atau tempel saat diminta).
2. Klik **Runtime → Run all**.
3. Setelah selesai, kembali ke admin **Settings → Models**, pilih provider
   **Dropbox**, lalu tekan **Scan** dan **Aktifkan** model.

Upload memakai kuota/akun Dropbox milik Anda sendiri.


In [ ]:
# @title Konfigurasi model { display-mode: "form" }
MODEL_REPO = "Systran/faster-whisper-large-v3"  # @param ["Systran/faster-whisper-large-v3", "Systran/faster-whisper-large-v3-turbo", "Systran/faster-whisper-medium", "Systran/faster-whisper-small", "openai/whisper-large-v3"]
SLUG = "large-v3"  # @param {type:"string"}
QUANTIZATION = "float16"  # @param ["float16", "int8", "int8_float16"]
OVERWRITE = False  # @param {type:"boolean"}
RESUME = True  # @param {type:"boolean"}

DROPBOX_ROOT = "/Clipzy/whisper_models"  # @param {type:"string"}
CHUNK_MB = 8  # @param {type:"integer"}



In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-whisper",
        "ctranslate2",
        "transformers",
        "huggingface_hub",
        "tqdm",
        "dropbox",
    ]
)


In [ ]:
from getpass import getpass

import dropbox

APP_KEY = getpass("Dropbox App key: ")
APP_SECRET = getpass("Dropbox App secret: ")

flow = dropbox.DropboxOAuth2FlowNoRedirect(
    APP_KEY, APP_SECRET, token_access_type="offline"
)
print("Buka URL berikut, izinkan akses, lalu tempel kodenya:")
print(flow.start())
auth_code = input("Kode otorisasi: ").strip()
result = flow.finish(auth_code)
print("REFRESH TOKEN (simpan di admin Clipzi):", result.refresh_token)

dbx = dropbox.Dropbox(
    oauth2_refresh_token=result.refresh_token, app_key=APP_KEY, app_secret=APP_SECRET
)
print("Terhubung sebagai:", dbx.users_get_current_account().email)


In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download

WORK = Path("/content/clipzi_model")
SOURCE = WORK / "source"
CT2_DIR = WORK / "ct2"
MODEL_DIR = WORK / "upload"

if WORK.exists():
    shutil.rmtree(WORK)
SOURCE.mkdir(parents=True, exist_ok=True)

print("Mengunduh:", MODEL_REPO)
snapshot_download(repo_id=MODEL_REPO, local_dir=str(SOURCE))
print("Download selesai:", SOURCE)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REQUIRED = ("model.bin", "config.json", "tokenizer.json", "preprocessor_config.json")
VOCAB = ("vocabulary.json", "vocabulary.txt")
PYTORCH = ("model.safetensors", "pytorch_model.bin")


def names(path: Path) -> set[str]:
    return {item.name.lower() for item in path.iterdir() if item.is_file()}


found = names(SOURCE)
print("File HF:", sorted(found)[:12])

if "model.bin" in found:
    TARGET = SOURCE
else:
    if not any(marker in found for marker in PYTORCH):
        raise SystemExit(
            "Repo tidak berisi model.bin (CT2) maupun bobot PyTorch. "
            "Pakai repo Systran/faster-whisper-* atau openai/whisper-*."
        )
    print("Konversi PyTorch -> CTranslate2 (float16)…")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "ctranslate2.converters.transformers",
            "--model",
            str(SOURCE),
            "--quantization",
            QUANTIZATION,
            "--output_dir",
            str(CT2_DIR),
        ]
    )
    TARGET = CT2_DIR

present = names(TARGET)
missing = [name for name in REQUIRED if name not in present]
if missing:
    raise SystemExit(f"Model CT2 tidak lengkap, kurang: {missing}")
if not any(name in present for name in VOCAB):
    raise SystemExit("Model CT2 tidak punya vocabulary.json/vocabulary.txt.")

size = sum(item.stat().st_size for item in TARGET.iterdir() if item.is_file())
print(f"Model siap: {TARGET} ({size / (1024**3):.2f} GiB)")


In [ ]:
from pathlib import Path

import dropbox

ROOT = DROPBOX_ROOT.rstrip("/")
DEST = f"{ROOT}/{SLUG}"
try:
    dbx.files_create_folder_v2(DEST)
except Exception as exc:  # folder sudah ada
    if "conflict" not in str(exc).lower():
        raise

CHUNK = int(CHUNK_MB) * 1024 * 1024
for path in sorted(TARGET.iterdir()):
    if not path.is_file():
        continue
    target = f"{DEST}/{path.name}"
    size = path.stat().st_size
    if not OVERWRITE:
        try:
            meta = dbx.files_get_metadata(target)
            if getattr(meta, "size", None) == size:
                print("skip (sudah sama):", target)
                continue
        except Exception:
            pass
    print(f"upload {path.name} ({size / 1048576:.1f} MiB)")
    if size <= CHUNK:
        with open(path, "rb") as handle:
            dbx.files_upload(
                handle.read(), target, mode=dropbox.files.WriteMode.overwrite
            )
        continue
    with open(path, "rb") as handle:
        session = dbx.files_upload_session_start(handle.read(CHUNK))
        cursor = dropbox.files.UploadSessionCursor(
            session_id=session.session_id, offset=handle.tell()
        )
        while handle.tell() < size:
            if size - handle.tell() <= CHUNK:
                commit = dropbox.files.CommitInfo(
                    path=target, mode=dropbox.files.WriteMode.overwrite
                )
                dbx.files_upload_session_finish(handle.read(CHUNK), cursor, commit)
                break
            dbx.files_upload_session_append_v2(handle.read(CHUNK), cursor)
            cursor.offset = handle.tell()
print("Selesai:", DEST)
